# 03 — Leakage-Free Splitting & Reproducible Preprocessing Pipeline

Verifies deduplicated stratified 80/20 splitting, zero train/test feature overlap, `log1p` skewness handling, median imputation, scaling, binary gender encoding, and in-fold resampling safety.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import pandas as pd
import numpy as np
from IPython.display import display, Image
from liver_disease.config import PROCESSED_DATA_DIR
from liver_disease.data.loader import load_raw_dataset
from liver_disease.data.preprocessing import (
    build_preprocessor,
    build_training_pipeline,
    create_leakage_free_split,
    get_transformed_feature_names,
)
from liver_disease.utils.io import load_json

df_raw = load_raw_dataset()
X_train, X_test, y_train, y_test, split_meta = create_leakage_free_split(df_raw, save_artifacts=False)
print('Train shape:', X_train.shape, '| Positive rate:', round(float(y_train.mean()), 4))
print('Test shape: ', X_test.shape, '| Positive rate:', round(float(y_test.mean()), 4))
print('Train/Test exact feature overlap:', split_meta['train_test_feature_overlap'])

Train shape: (456, 10) | Positive rate: 0.7127
Test shape:  (114, 10) | Positive rate: 0.7105
Train/Test exact feature overlap: 0


In [2]:
pre = build_preprocessor(scaler='robust', add_clinical_ratios=False, log_transform_skewed=True)
X_train_t = pre.fit_transform(X_train)
X_test_t = pre.transform(X_test)
feat_names = get_transformed_feature_names(pre)
print('Transformed feature names:', feat_names)
print('Train NaNs:', int(np.isnan(X_train_t).sum()), '| Test NaNs:', int(np.isnan(X_test_t).sum()))
display(pd.DataFrame(X_train_t, columns=feat_names).describe().T[['mean', 'std', 'min', '50%', 'max']].round(3))

Transformed feature names: ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'A/G Ratio', 'Gender_Male']
Train NaNs: 0 | Test NaNs: 0


,mean,std,min,50%,max
Age,-0.013,0.656,-1.640,0.0,1.800
TB,0.532,1.146,-0.581,0.0,5.928
DB,0.521,1.029,-0.276,0.0,4.451
Alkphos,0.288,1.021,-1.980,0.0,4.557
Sgpt,0.203,0.978,-1.318,0.0,4.467
Sgot,0.191,0.820,-1.119,0.0,3.936
TP,-0.079,0.826,-2.923,0.0,2.308
ALB,0.047,0.654,-1.750,0.0,2.000
A/G Ratio,-0.037,0.765,-1.662,0.0,3.837
Gender_Male,0.765,0.424,0.000,1.0,1.000
